<a href="https://colab.research.google.com/github/mariaaapetrovskaya/complingua/blob/main/%D0%9A%D0%BE%D0%BF%D0%B8%D1%8F_%D0%B1%D0%BB%D0%BE%D0%BA%D0%BD%D0%BE%D1%82%D0%B0_%2214_03_26_%22fine_tuning_hw_ipynb%22%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets evaluate accelerate -q

import torch
print(f"GPUдоступен: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

import numpy as np
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from datasets import load_dataset
import evaluate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.0 MB/s eta 0:00:00
GPUдоступен: True
GPU: Tesla T4


In [ ]:
dataset = load_dataset("fancyzhx/ag_news")
print(f"Датасет загружен. Train: {len(dataset['train'])}, Test: {len(dataset['test'])}")

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=4
).to(device)

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)
train_dataset = tokenized_datasets["train"]
test_dataset = tokenized_datasets["test"]

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [ ]:
training_args = TrainingArguments(
    output_dir="./results-ag_news",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="tensorboard",
    logging_steps=500,
)


In [ ]:
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy_metric.compute(predictions=predictions, references=labels)


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy
1,0.197523,0.181552,0.941974
2,0.133741,0.192665,0.946842
3,0.089417,0.218588,0.947500


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=22500, training_loss=0.154226245964898, metrics={'train_runtime': 3335.9927, 'train_samples_per_second': 107.914, 'train_steps_per_second': 6.745, 'total_flos': 8558812764910464.0, 'train_loss': 0.154226245964898, 'epoch': 3.0})

In [ ]:
eval_results = trainer.evaluate()
print(f"\nEvaluation results: {eval_results}")



Evaluation results: {'eval_loss': 0.21858790516853333, 'eval_accuracy': 0.9475, 'eval_runtime': 22.5093, 'eval_samples_per_second': 337.638, 'eval_steps_per_second': 21.102, 'epoch': 3.0}


In [ ]:
model.save_pretrained("./ag_news_model")
tokenizer.save_pretrained("./ag_news_model")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./ag_news_model/tokenizer_config.json', './ag_news_model/tokenizer.json')

In [ ]:
from transformers import pipeline

classifier = pipeline("text-classification", model="./ag_news_model", tokenizer="./ag_news_model")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [ ]:
test_news = [
    "TikTok Investors Are Set to Pay $10 Billion Fee to Trump Administration The large fee is the latest example of the White House’s inserting itself into corporate deal making in unusual and aggressive ways",
    "Explosion at Amsterdam Jewish school 'a deliberate attack' says mayor Security had been increased at Jewish public institutions following an earlier incident in Rotterdam.",
    "New York Academy of Art Gives Away Money Donated by Jeffrey Epstein The school also said it would review policies about philanthropy and donor engagement after new revelations about the disgraced financier were made public"
]

for text in test_news:
    result = classifier(text)[0]
    print(f"Text: {text}\nLabel: {result['label']}, Score: {result['score']:.4f}\n")

Text: TikTok Investors Are Set to Pay $10 Billion Fee to Trump Administration The large fee is the latest example of the White House’s inserting itself into corporate deal making in unusual and aggressive ways
Label: LABEL_2, Score: 0.7539

Text: Explosion at Amsterdam Jewish school 'a deliberate attack' says mayor Security had been increased at Jewish public institutions following an earlier incident in Rotterdam.
Label: LABEL_0, Score: 0.9993

Text: New York Academy of Art Gives Away Money Donated by Jeffrey Epstein The school also said it would review policies about philanthropy and donor engagement after new revelations about the disgraced financier were made public
Label: LABEL_2, Score: 0.9972

